In [1]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os


In [2]:
load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("ocean-knowledge")

# Fetch the anchovy spawning chunk directly
result = index.fetch(ids=["anchovy_general_spawning_1"])
print(result)


/Users/roshnij/Desktop/floatchat/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FetchResponse(namespace='', vectors={}, usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 18 Apr 2026 05:44:45 GMT', 'content-type': 'application/json', 'content-length': '53', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '191', 'x-envoy-upstream-service-time': '191', 'x-pinecone-response-duration-ms': '192', 'grpc-status': '0', 'server': 'envoy'}})


In [17]:
questions=["What do Indian Mackerel eat?"]

In [18]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

load_dotenv()

pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("ocean-knowledge")
model = SentenceTransformer("all-MiniLM-L6-v2")

# questions = [
#     "What do Indian Mackerel eat?",
#     "When does anchovy feeding peak during the year?",
#     "What is the diet of Black-banded Trevally?",
#     "Do sardines feed on phytoplankton or zooplankton?",
#     "When do anchovies spawn in the Arabian Sea?",
#     "What is the spawning season of Indian Mackerel?",
#     "At what size does Thryssa mystax reach maturity?",
#     "When does Hilsa migrate upstream to spawn?",
#     "What depth does Indian Squid live at?",
#     "What temperature does Yellowfin Tuna prefer?",
#     "What salinity range can Spiny Lobster survive in?",
#     "Where does Silver Pomfret live on the seafloor?",
#     "What is the best month to catch sardines on the west coast?",
#     "When is peak season for prawns in the Arabian Sea?",
#     "When is seer fish most available in Bay of Bengal?",
#     "What causes upwelling in the Arabian Sea?",
#     "What is the oxygen minimum zone?",
#     "How does the southwest monsoon affect chlorophyll levels?",
#     "What happens to salinity in Bay of Bengal during monsoon?",
#     "When is the trawl ban on the Karnataka coast?",
# ]

print("=" * 60)

for q in questions:
    embedding = model.encode([q]).tolist()[0]

    result = index.query(
        vector=embedding,
        top_k=1,
        include_metadata=True
    )

    if result["matches"]:
        match  = result["matches"][0]
        doc    = match["metadata"].get("document_text", "")[:120]
        score  = match["score"]
        mid    = match["id"]
        mtype  = match["metadata"].get("type", "unknown")
        print(f"\nQ: {q}")
        print(f"   ID    : {mid}")
        print(f"   Type  : {mtype}  |  Score: {score:.3f}")
        print(f"   Text  : {doc}...")
    else:
        print(f"\nQ: {q}")
        print(f"   ❌ No match found")

print("\n" + "=" * 60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6718.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Q: What do Indian Mackerel eat?
   ID    : mackerel_general_fishery_7
   Type  : fishery  |  Score: 0.744
   Text  : Indian mackerel (Rastrelliger kanagurta) is a highly commercial species marketed fresh, frozen, canned, dried-salted, sm...



In [12]:
result = index.query(
    vector=model.encode(["Indian Mackerel feeding diet"]).tolist()[0],
    top_k=10,
    include_metadata=True
)

for match in result["matches"]:
    print(f"  ID    : {match['id']}")
    print(f"  Type  : {match['metadata'].get('type')}")
    print(f"  Species: {match['metadata'].get('species')}")
    print(f"  Score : {match['score']:.3f}")
    print()

  ID    : mackerel_general_fishery_7
  Type  : fishery
  Species: mackerel
  Score : 0.759

  ID    : indian_mackerel_general_fishery_5
  Type  : fishery
  Species: indian mackerel
  Score : 0.714

  ID    : mackerel_general_feeding_5
  Type  : feeding
  Species: mackerel
  Score : 0.682

  ID    : mackerel_general_biology_3
  Type  : biology
  Species: mackerel
  Score : 0.633

  ID    : indian_mackerel_general_feeding_4
  Type  : feeding
  Species: indian mackerel
  Score : 0.616

  ID    : mackerel_general_habitat_2
  Type  : habitat
  Species: mackerel
  Score : 0.601

  ID    : indian_mackerel_general_habitat_1
  Type  : habitat
  Species: indian mackerel
  Score : 0.574

  ID    : mackerel_general_taxonomy_1
  Type  : taxonomy
  Species: mackerel
  Score : 0.569

  ID    : indian_mackerel_bay_of_bengal_fishery_3
  Type  : fishery
  Species: indian mackerel
  Score : 0.568

  ID    : mackerel_general_distribution_4
  Type  : habitat
  Species: mackerel
  Score : 0.552



In [13]:
import json

SPECIES_MAP = {
    "indian anchovy": "anchovy",
    "indian mackerel": "mackerel",
    "skipjack tuna": "skipjack_tuna",
    "yellowfin tuna": "yellowfin_tuna",
    "seer fish": "seer_fish",
    "black-banded trevally": "black_banded_trevally",
    "scalloped spiny lobster": "spiny_lobster",
}

with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

seen_ids = set()
cleaned = []
duplicates = 0

for chunk in chunks:
    # Fix duplicate IDs
    chunk_id = chunk.get("id", "")
    if chunk_id in seen_ids:
        print(f"  Removed duplicate: {chunk_id}")
        duplicates += 1
        continue
    seen_ids.add(chunk_id)

    # Fix species names
    species = chunk.get("metadata", {}).get("species", "")
    if species in SPECIES_MAP:
        chunk["metadata"]["species"] = SPECIES_MAP[species]
        print(f"  Fixed species: {species} → {SPECIES_MAP[species]}")

    cleaned.append(chunk)

with open("chunks_clean.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)

print(f"\n✅ Done")
print(f"   Original chunks : {len(chunks)}")
print(f"   Duplicates removed: {duplicates}")
print(f"   Clean chunks : {len(cleaned)}")
print(f"   Saved to: chunks_clean.json")

  Fixed species: black-banded trevally → black_banded_trevally
  Fixed species: black-banded trevally → black_banded_trevally
  Fixed species: black-banded trevally → black_banded_trevally
  Fixed species: scalloped spiny lobster → spiny_lobster
  Fixed species: indian anchovy → anchovy
  Fixed species: indian anchovy → anchovy
  Fixed species: indian anchovy → anchovy
  Fixed species: indian anchovy → anchovy
  Fixed species: indian anchovy → anchovy
  Removed duplicate: hilsa_bay_of_bengal_behavior_4
  Fixed species: indian mackerel → mackerel
  Fixed species: indian mackerel → mackerel
  Fixed species: indian mackerel → mackerel
  Fixed species: indian mackerel → mackerel
  Fixed species: indian mackerel → mackerel
  Fixed species: seer fish → seer_fish
  Fixed species: seer fish → seer_fish
  Fixed species: seer fish → seer_fish
  Fixed species: seer fish → seer_fish
  Fixed species: seer fish → seer_fish
  Fixed species: skipjack tuna → skipjack_tuna
  Fixed species: skipjack tuna

In [16]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

load_dotenv()

pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("ocean-knowledge")
model = SentenceTransformer("all-MiniLM-L6-v2")

query = "What do Indian Mackerel eat?"
embedding = model.encode([query]).tolist()[0]

# Query WITHOUT any filter — see raw scores
print("=== NO FILTER ===")
results = index.query(
    vector=embedding,
    top_k=5,
    include_metadata=True
)
for m in results["matches"]:
    print(f"  {m['id']}")
    print(f"  species={m['metadata'].get('species')} type={m['metadata'].get('type')} score={m['score']:.3f}")
    print()

# Query WITH type filter
print("=== FILTER: type=feeding ===")
results = index.query(
    vector=embedding,
    top_k=3,
    filter={"type": {"$eq": "feeding"}},
    include_metadata=True
)
for m in results["matches"]:
    print(f"  {m['id']}")
    print(f"  species={m['metadata'].get('species')} type={m['metadata'].get('type')} score={m['score']:.3f}")
    print()

# Query WITH species + type filter
print("=== FILTER: species=mackerel AND type=feeding ===")
results = index.query(
    vector=embedding,
    top_k=3,
    filter={
        "species": {"$eq": "mackerel"},
        "type": {"$eq": "feeding"}
    },
    include_metadata=True
)
for m in results["matches"]:
    print(f"  {m['id']}")
    print(f"  species={m['metadata'].get('species')} type={m['metadata'].get('type')} score={m['score']:.3f}")
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5796.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== NO FILTER ===
  mackerel_general_fishery_7
  species=mackerel type=fishery score=0.744

  indian_mackerel_general_fishery_5
  species=mackerel type=fishery score=0.730

  mackerel_general_habitat_2
  species=mackerel type=habitat score=0.660

  indian_mackerel_general_habitat_1
  species=mackerel type=habitat score=0.647

  mackerel_general_feeding_5
  species=mackerel type=feeding score=0.645

=== FILTER: type=feeding ===
  mackerel_general_feeding_5
  species=mackerel type=feeding score=0.645

  indian_mackerel_general_feeding_4
  species=mackerel type=feeding score=0.599

  sardine_general_feeding_5
  species=sardine type=feeding score=0.500

=== FILTER: species=mackerel AND type=feeding ===
  mackerel_general_feeding_5
  species=mackerel type=feeding score=0.645

  indian_mackerel_general_feeding_4
  species=mackerel type=feeding score=0.599



In [19]:
stats = index.describe_index_stats()

In [20]:
print(stats["total_vector_count"])

175


In [21]:
print(stats["namespaces"])

{'__default__': {'vector_count': 175}}


In [23]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

load_dotenv()

pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("ocean-knowledge")
model = SentenceTransformer("all-MiniLM-L6-v2")

# 👉 Example filters (change as needed)
FILTER = {
    "species": "pomfret",
    "type": "feeding"
}

print("=" * 60)

for q in questions:
    embedding = model.encode([q]).tolist()[0]

    result = index.query(
        vector=embedding,
        top_k=1,
        include_metadata=True,
        filter=FILTER   # ✅ FILTER ADDED HERE
    )

    if result["matches"]:
        match  = result["matches"][0]
        doc    = match["metadata"].get("document_text", "")[:120]
        score  = match["score"]
        mid    = match["id"]
        mtype  = match["metadata"].get("type", "unknown")

        print(f"\nQ: {q}")
        print(f"   ID    : {mid}")
        print(f"   Type  : {mtype}  |  Score: {score:.3f}")
        print(f"   Text  : {doc}...")
    else:
        print(f"\nQ: {q}")
        print(f"   ❌ No match found")

print("\n" + "=" * 60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7027.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Q: What do Indian Mackerel eat?
   ID    : pomfret_general_feeding_2
   Type  : feeding  |  Score: 0.438
   Text  : Pomfret forms dense schools over muddy and sandy bottom habitats within the coastal-shelf benthopelagic zone. Adults fee...



In [24]:
"""
pinecone_diagnostic.py
Run this ONCE before using the engine to verify:
  1. What metadata keys Pinecone actually stored
  2. What text field name holds the chunk content
  3. Whether $eq filters return results
  4. What Neon actually contains (depth zones, regions, date range)

Usage:
    python pinecone_diagnostic.py
"""

import os
from dotenv import load_dotenv
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()

# ── Pinecone ──────────────────────────────────────────────────────────────────
pc    = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(os.getenv("PINECONE_INDEX_NAME", "ocean-knowledge"))
emb   = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("\n" + "="*60)
print("STEP 1: Fetch a known chunk by ID to inspect raw metadata")
print("="*60)
# These IDs come from your knowledge base JSON
KNOWN_IDS = [
    "anchovy_general_feeding_1",
    "indian_mackerel_general_feeding_4",
    "hilsa_bay_of_bengal_fishery_8",
    "ocean_arabian_sea_upwelling_2",
]
for chunk_id in KNOWN_IDS:
    try:
        result = index.fetch(ids=[chunk_id])
        vectors = result.get("vectors", {})
        if chunk_id in vectors:
            meta = vectors[chunk_id].get("metadata", {})
            print(f"\n  ID: {chunk_id}")
            print(f"  Metadata keys: {list(meta.keys())}")
            for k, v in meta.items():
                display = str(v)[:120] + "..." if len(str(v)) > 120 else str(v)
                print(f"    {k!r}: {display!r}")
        else:
            print(f"\n  ID: {chunk_id} → NOT FOUND in index")
    except Exception as e:
        print(f"\n  ID: {chunk_id} → Error: {e}")

print("\n" + "="*60)
print("STEP 2: Unfiltered semantic query — verify text field name")
print("="*60)
test_vec = emb.embed_query("what do mackerel eat")
raw = index.query(vector=test_vec, top_k=2, include_metadata=True)
for i, match in enumerate(raw.get("matches", [])):
    meta = match.get("metadata", {})
    print(f"\n  Match {i+1} | score={match['score']:.4f}")
    print(f"  Metadata keys: {list(meta.keys())}")
    # Try every plausible text key
    for candidate_key in ["document", "text", "content", "chunk", "page_content"]:
        val = meta.get(candidate_key, "")
        if val:
            print(f"  ✓ Text found under key '{candidate_key}': {val[:100]}...")
            break
    else:
        print("  ✗ No text content found under any standard key")

print("\n" + "="*60)
print("STEP 3: Test $eq filter on species")
print("="*60)
for species_val in ["mackerel", "hilsa", "anchovy"]:
    try:
        r = index.query(
            vector=test_vec,
            top_k=2,
            include_metadata=True,
            filter={"species": {"$eq": species_val}},
        )
        count = len(r.get("matches", []))
        print(f"  species=$eq'{species_val}' → {count} results")
        if count > 0:
            print(f"    Sample type: {r['matches'][0]['metadata'].get('type', 'N/A')}")
    except Exception as e:
        print(f"  species=$eq'{species_val}' → ERROR: {e}")

print("\n" + "="*60)
print("STEP 4: Test $eq filter on type")
print("="*60)
for type_val in ["feeding", "habitat", "fishery", "oceanography"]:
    try:
        r = index.query(
            vector=test_vec,
            top_k=2,
            include_metadata=True,
            filter={"type": {"$eq": type_val}},
        )
        count = len(r.get("matches", []))
        print(f"  type=$eq'{type_val}' → {count} results")
    except Exception as e:
        print(f"  type=$eq'{type_val}' → ERROR: {e}")

print("\n" + "="*60)
print("STEP 5: Test $in filter on regions")
print("="*60)
for region_val in ["Arabian Sea", "Bay of Bengal"]:
    try:
        r = index.query(
            vector=test_vec,
            top_k=2,
            include_metadata=True,
            filter={"regions": {"$in": [region_val]}},
        )
        count = len(r.get("matches", []))
        print(f"  regions=$in['{region_val}'] → {count} results")
    except Exception as e:
        print(f"  regions=$in['{region_val}'] → ERROR: {e}")

print("\n" + "="*60)
print("STEP 6: Combined filter — species + type + region")
print("="*60)
try:
    r = index.query(
        vector=emb.embed_query("what does mackerel eat in the Arabian Sea"),
        top_k=3,
        include_metadata=True,
        filter={
            "species": {"$eq": "mackerel"},
            "type":    {"$eq": "feeding"},
            "regions": {"$in": ["Arabian Sea"]},
        },
    )
    count = len(r.get("matches", []))
    print(f"  Combined filter → {count} results")
    for m in r.get("matches", []):
        meta = m["metadata"]
        print(f"    score={m['score']:.4f} | species={meta.get('species')} | "
              f"type={meta.get('type')} | regions={meta.get('regions')}")
except Exception as e:
    print(f"  Combined filter → ERROR: {e}")

print("\n" + "="*60)
print("STEP 7: Neon DB — actual depth zones, regions, date range")
print("="*60)
try:
    engine = create_engine(
        os.getenv("DATABASE_URL"),
        connect_args={"sslmode": "require"},
    )
    with engine.connect() as conn:
        # Distinct depth zones
        df_depths = pd.read_sql(
            text("SELECT DISTINCT depth_zone FROM argo_ocean_data ORDER BY depth_zone;"),
            conn,
        )
        print(f"\n  depth_zone values in DB: {df_depths['depth_zone'].tolist()}")

        # Distinct regions
        df_regions = pd.read_sql(
            text("SELECT DISTINCT region_name FROM argo_ocean_data ORDER BY region_name;"),
            conn,
        )
        print(f"  region_name values in DB: {df_regions['region_name'].tolist()}")

        # Date range
        df_range = pd.read_sql(
            text("SELECT MIN(year) as min_yr, MAX(year) as max_yr, "
                 "MIN(month) as min_mo, MAX(month) as max_mo "
                 "FROM argo_ocean_data;"),
            conn,
        )
        row = df_range.iloc[0]
        print(f"  Year range: {int(row['min_yr'])} – {int(row['max_yr'])}")
        print(f"  Month range: {int(row['min_mo'])} – {int(row['max_mo'])}")

        # Row counts per region+depth
        df_counts = pd.read_sql(
            text("SELECT region_name, depth_zone, COUNT(*) as n "
                 "FROM argo_ocean_data "
                 "GROUP BY region_name, depth_zone "
                 "ORDER BY region_name, depth_zone;"),
            conn,
        )
        print("\n  Row counts per (region, depth):")
        print(df_counts.to_string(index=False))

        # Sample recent rows
        df_sample = pd.read_sql(
            text("SELECT region_name, depth_zone, year, month, "
                 "avg_temp_celsius, avg_salinity_psu "
                 "FROM argo_ocean_data "
                 "ORDER BY year DESC, month DESC LIMIT 5;"),
            conn,
        )
        print("\n  5 most recent rows:")
        print(df_sample.to_string(index=False))

except Exception as e:
    print(f"  Neon connection error: {e}")

print("\n" + "="*60)
print("DIAGNOSTIC COMPLETE")
print("Share the output above to identify the exact cause of 0 results.")
print("="*60 + "\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6384.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 1: Fetch a known chunk by ID to inspect raw metadata

  ID: anchovy_general_feeding_1 → NOT FOUND in index

  ID: indian_mackerel_general_feeding_4
  Metadata keys: ['associated_features', 'behavior', 'diet', 'document_text', 'region_specificity', 'regions', 'scientific_names', 'species', 'tags', 'type']
    'associated_features': "['high chlorophyll', 'phytoplankton blooms', 'upwelling', 'zooplankton availability', 'coastal productivity']"
    'behavior': "['schooling', 'surface feeding', 'mixed filter and particulate feeding', 'feeding linked to plankton blooms']"
    'diet': "['phytoplankton', 'diatoms', 'zooplankton', 'cladocerans', 'ostracods', 'polychaete larvae', 'larval shrimp', 'fish eggs..."
    'document_text': 'Indian mackerel is an obligate schooling species that forms large, dense schools, often detected near the surface during...'
    'region_specificity': 'general'
    'regions': "['Arabian Sea', 'Bay of Bengal']"
    'scientific_names': "['Rastrelliger kanagurta'